In [ ]:
from keras.models import Sequential, Model
import keras.backend as K
from keras.layers import GRU, Dense, TimeDistributed, Lambda, Flatten, Dropout, Conv1D, BatchNormalization, ReLU, MaxPooling1D, Masking, LayerNormalization
from keras.layers import Input, Reshape, concatenate, Permute, Multiply, RepeatVector, Embedding, GlobalAveragePooling1D, Activation
from keras.optimizers import Adam
from keras.models import Model
import tensorflow as tf
from keras.regularizers import l2

In [ ]:
class BaseGRU:
    def __init__(self, input_shape, learning_rate, dropout, recurrent_dropout,
                 kernel_regularizer, recurrent_regularizer):
        self.input_shape = input_shape
        self.learning_rate = learning_rate
        self.dropout = dropout
        self.recurrent_dropout = recurrent_dropout
        self.kernel_regularizer = kernel_regularizer
        self.recurrent_regularizer = recurrent_regularizer
        self.model = self.create_model()

    def create_model(self):
        model = Sequential()
        model.add(GRU(20, dropout=self.dropout, recurrent_dropout=self.recurrent_dropout, return_sequences=True,
                      input_shape=self.input_shape, kernel_regularizer=self.kernel_regularizer, recurrent_regularizer=self.recurrent_regularizer))
        model.add(TimeDistributed(Dense(500, activation='tanh', kernel_regularizer=self.kernel_regularizer)))
        model.add(TimeDistributed(Dense(1, activation='sigmoid', kernel_regularizer=self.kernel_regularizer)))
        model.add(Flatten())

        def mil_aggregation(x):
            return tf.reduce_max(x, axis=1, keepdims=True)

        model.add(Lambda(mil_aggregation, output_shape=(1,)))

        model.compile(optimizer=Adam(learning_rate=self.learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
        return model

    def summary(self):
        return self.model.summary()

    def fit(self, x, y, epochs=10, batch_size=32, validation_data=None):
        return self.model.fit(x, y, epochs=epochs, batch_size=batch_size, validation_data=validation_data)

    def evaluate(self, x, y):
        return self.model.evaluate(x, y)

    def predict(self, x):
        return self.model.predict(x)

    def save(self, filepath):
        self.model.save(filepath)

In [ ]:
class MultiHeadCnnRnn:
    def __init__(self, input_shape, kernel_sizes, filters, learning_rate, weight_decay,
                 dropout, recurrent_dropout, kernel_regularizer, recurrent_regularizer):
        self.input_shape = input_shape
        self.kernel_sizes = kernel_sizes
        self.filters = filters
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.dropout = dropout
        self.recurrent_dropout = recurrent_dropout
        self.kernel_regularizer = kernel_regularizer
        self.recurrent_regularizer = recurrent_regularizer
        self.model = self.create_model()

    def create_model(self):
        input_layer = Input(shape=self.input_shape)

        # Multi-Head CNN processing
        cnn_outputs = []
        print(f'Feature amount: {self.input_shape[1]}')
        for i in range(self.input_shape[1]):
            head = Reshape((self.input_shape[0], 1))(input_layer[:, :, i:i+1])
            for j in range(len(self.kernel_sizes)):
                head = Conv1D(filters=self.filters[j], kernel_size=self.kernel_sizes[j], activation='relu',
                              kernel_regularizer=l2(self.weight_decay))(head)
                head = BatchNormalization()(head)
            head = Conv1D(filters=1, kernel_size=1, padding='same', activation='sigmoid',
                          kernel_regularizer=l2(self.weight_decay))(head)
            cnn_outputs.append(head)
        concatenated = concatenate(cnn_outputs)

        # Sequential GRU processing from BaseGRU configuration
        gru_layer = GRU(20, dropout=self.dropout, recurrent_dropout=self.recurrent_dropout,
                        return_sequences=True, kernel_regularizer=self.kernel_regularizer,
                        recurrent_regularizer=self.recurrent_regularizer)(concatenated)
        time_distributed_dense = TimeDistributed(Dense(100, activation='tanh', kernel_regularizer=self.kernel_regularizer))(gru_layer)
        time_distributed_output = TimeDistributed(Dense(1, activation='sigmoid', kernel_regularizer=self.kernel_regularizer))(time_distributed_dense)
        flattened = Flatten()(time_distributed_output)

        # Custom MIL aggregation layer using max functions
        def mil_aggregation(x):
            return tf.reduce_max(x, axis=1, keepdims=True)

        mil_output = Lambda(mil_aggregation, output_shape=(1,))(flattened)

        # Compile the model
        model = Model(inputs=input_layer, outputs=mil_output)
        model.compile(optimizer=Adam(learning_rate=self.learning_rate), loss='binary_crossentropy', metrics=['accuracy'])

        return model

    def summary(self):
        return self.model.summary()

    def fit(self, x, y, epochs=30, batch_size=128, validation_data=None):
        return self.model.fit(x, y, epochs=epochs, batch_size=batch_size, validation_data=validation_data)

    def evaluate(self, x, y):
        return self.model.evaluate(x, y)

    def predict(self, x):
        return self.model.predict(x)

    def save(self, filepath):
        self.model.save(filepath)

In [ ]:
class MultiHeadAttention:
    def __init__(self, input_shape, kernel_sizes, filters, learning_rate, weight_decay,
                 dropout, recurrent_dropout, kernel_regularizer, recurrent_regularizer,
                 attention_block_type):
        """
        attention_block_type: String, either 'block1' or 'block2', to choose the attention mechanism.
        """
        self.input_shape = input_shape
        self.kernel_sizes = kernel_sizes
        self.filters = filters
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.dropout = dropout
        self.recurrent_dropout = recurrent_dropout
        self.kernel_regularizer = kernel_regularizer
        self.recurrent_regularizer = recurrent_regularizer
        self.attention_block_type = attention_block_type
        self.model = self.create_model()

    def attention_3d_block1(self, inputs, single_attention_vector=False):
        """
        Attention Block 1: Applies a Dense layer with softmax activation directly on the inputs.
        It learns attention weights over the feature dimension at each time step.
        """
        input_dim = int(inputs.shape[2])
        # Apply Dense layer on inputs to learn attention weights
        a = Dense(input_dim, activation='softmax')(inputs)
        if single_attention_vector:
            # Average across time steps to obtain a single attention vector
            a = Lambda(lambda x: K.mean(x, axis=1), name='dim_reduction')(a)
            a = RepeatVector(input_dim)(a)
        # Permute so that shape is consistent, though here it remains (batch, time_steps, input_dim)
        a_probs = Permute((1, 2), name='attention_vec')(a)
        # Multiply original inputs by the attention weights
        output_attention_mul = Multiply()([inputs, a_probs])
        return output_attention_mul

    def attention_3d_block2(self, inputs, single_attention_vector=False):
        """
        Attention Block 2: Permutes the inputs so that the Dense layer learns attention over time steps.
        This is useful when time-step importance is desired.
        """
        time_steps = int(inputs.shape[1])
        input_dim = int(inputs.shape[2])
        # Permute dimensions to (batch_size, input_dim, time_steps)
        a = Permute((2, 1))(inputs)
        # Learn attention weights with a Dense layer and softmax activation over time steps.
        a = Dense(time_steps, activation='softmax')(a)
        if single_attention_vector:
            # Average over the feature dimension to get a single attention vector for all features.
            a = Lambda(lambda x: K.mean(x, axis=1), name='dim_reduction')(a)
            a = RepeatVector(input_dim)(a)
        # Permute back to the original shape: (batch_size, time_steps, input_dim)
        a_probs = Permute((2, 1))(a)
        # Multiply input by the attention weights
        output_attention_mul = Multiply()([inputs, a_probs])
        return output_attention_mul

    def create_model(self):
        input_layer = Input(shape=self.input_shape)

        # Multi-Head CNN processing: Process each feature channel separately.
        cnn_outputs = []
        # For each feature (column) in the input
        for i in range(self.input_shape[1]):
            # Extract the i-th feature and reshape to (time_steps, 1)
            head = Reshape((self.input_shape[0], 1))(input_layer[:, :, i:i+1])
            # Apply a series of Conv1D and BatchNormalization layers per head.
            for j in range(len(self.kernel_sizes)):
                head = Conv1D(filters=self.filters[j],
                              kernel_size=self.kernel_sizes[j],
                              activation='relu',
                              padding='same',
                              kernel_regularizer=l2(self.weight_decay))(head)
                head = BatchNormalization()(head)
            # Final convolution to compress the head to a single channel with sigmoid activation.
            head = Conv1D(filters=1,
                          kernel_size=1,
                          padding='same',
                          activation='sigmoid',
                          kernel_regularizer=l2(self.weight_decay))(head)
            cnn_outputs.append(head)
        # Concatenate outputs from all heads along the feature dimension.
        concatenated = concatenate(cnn_outputs, axis=-1)

        # Sequential GRU processing from BaseGRU configuration.
        gru_layer = GRU(20, dropout=self.dropout, recurrent_dropout=self.recurrent_dropout,
                        return_sequences=True, kernel_regularizer=self.kernel_regularizer,
                        recurrent_regularizer=self.recurrent_regularizer)(concatenated)

        # Choose the attention block based on the provided parameter.
        if self.attention_block_type == 'block1':
            attention_out = self.attention_3d_block1(gru_layer, single_attention_vector=False)
        else:
            attention_out = self.attention_3d_block2(gru_layer, single_attention_vector=False)

        # Flatten the attention weighted output.
        flattened = Flatten()(attention_out)
        # Final dense layer for output (binary classification).
        output = Dense(1, activation='sigmoid', kernel_regularizer=self.kernel_regularizer)(flattened)

        model = Model(inputs=input_layer, outputs=output)
        model.compile(optimizer=Adam(learning_rate=self.learning_rate),
                      loss='binary_crossentropy',
                      metrics=['accuracy'])
        return model

    def summary(self):
        return self.model.summary()

    def fit(self, x, y, epochs=30, batch_size=128, validation_data=None):
        return self.model.fit(x, y, epochs=epochs, batch_size=batch_size, validation_data=validation_data)

    def evaluate(self, x, y):
        return self.model.evaluate(x, y)

    def predict(self, x):
        return self.model.predict(x)

    def save(self, filepath):
        self.model.save(filepath)

In [ ]:
# A simple positional encoding layer (similar to Vaswani et al., 2017)
class PositionalEncodingLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, max_len, **kwargs):
        super(PositionalEncodingLayer, self).__init__(**kwargs)
        self.d_model = d_model
        self.max_len = max_len

    def call(self, inputs):
        # inputs shape: (batch, seq_len, d_model)
        seq_len = tf.shape(inputs)[1]
        pos = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)
        i = tf.cast(tf.range(self.d_model)[tf.newaxis, :], tf.float32)
        angle_rates = 1 / tf.pow(10000.0, (2 * (i // 2)) / tf.cast(self.d_model, tf.float32))
        angle_rads = pos * angle_rates
        # apply sin to even indices; cos to odd indices
        sines = tf.math.sin(angle_rads[:, 0::2])
        cosines = tf.math.cos(angle_rads[:, 1::2])
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        pos_encoding = pos_encoding[tf.newaxis, ...]  # shape: (1, seq_len, d_model)
        return inputs + pos_encoding

    def compute_output_shape(self, input_shape):
        return input_shape

# The GTN model class
class GatedTransformerNetwork:
    def __init__(self, input_shape,
                 conv_filters=32,
                 conv_kernel_size=3,
                 embedding_dim=64,
                 transformer_layers=1,
                 num_heads=4,
                 dropout=0.2,
                 learning_rate=1e-4,
                 weight_decay=1e-4):
        self.input_shape = input_shape  # e.g. (81, 23)
        self.conv_filters = conv_filters
        self.conv_kernel_size = conv_kernel_size
        self.embedding_dim = embedding_dim
        self.transformer_layers = transformer_layers
        self.num_heads = num_heads
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.model = self.create_model()

    def create_model(self):
        # Input layer
        inputs = Input(shape=self.input_shape)  # shape: (timesteps, features)

        # ---- Convolutional Preprocessing ----
        # Apply a Conv1D layer to capture local temporal features
        conv_out = Conv1D(filters=self.conv_filters,
                          kernel_size=self.conv_kernel_size,
                          padding='same',
                          activation='relu',
                          kernel_regularizer=l2(self.weight_decay))(inputs)
        conv_out = BatchNormalization()(conv_out)
        # Project each time step to a higher-dimensional embedding with tanh activation
        preprocessed = TimeDistributed(Dense(self.embedding_dim, activation='tanh'))(conv_out)
        # Now preprocessed has shape (timesteps, embedding_dim)

        # ---- Step-wise (Temporal) Transformer Tower ----
        # Add positional encoding
        pos_encoded = PositionalEncodingLayer(self.embedding_dim, self.input_shape[0])(preprocessed)
        step_tower = pos_encoded
        # Apply a series of Transformer encoder blocks along the time dimension
        for _ in range(self.transformer_layers):
            # Multi-head self-attention
            attn_output = tf.keras.layers.MultiHeadAttention(
                num_heads=self.num_heads, key_dim=self.embedding_dim, dropout=self.dropout)(
                    step_tower, step_tower)
            attn_output = Dropout(self.dropout)(attn_output)
            step_tower = LayerNormalization(epsilon=1e-6)(step_tower + attn_output)
            # Feed-forward network
            ffn = Dense(self.embedding_dim * 4, activation='relu',
                        kernel_regularizer=l2(self.weight_decay))(step_tower)
            ffn = Dense(self.embedding_dim, kernel_regularizer=l2(self.weight_decay))(ffn)
            ffn = Dropout(self.dropout)(ffn)
            step_tower = LayerNormalization(epsilon=1e-6)(step_tower + ffn)
        # Aggregate over time (global average pooling)
        step_features = GlobalAveragePooling1D()(step_tower)
        # Fully connected layer to get the step-wise feature vector S
        S = Dense(self.embedding_dim, activation='tanh',
                  kernel_regularizer=l2(self.weight_decay))(step_features)

        # ---- Channel-wise Transformer Tower ----
        # For channel-wise processing, treat each channel as a token.
        # Here, we first transpose the raw input so that channels become the sequence tokens.
        # (No positional encoding is added because channel order is arbitrary.)
        # Note: we use the raw input here; you could also experiment with using the conv preprocessed branch.
        channel_input = Permute((2, 1))(inputs)  # shape: (features, timesteps)
        # Project each channel (i.e. the entire time series for that channel) to an embedding vector.
        channel_embed = Dense(self.embedding_dim, activation='tanh',
                              kernel_regularizer=l2(self.weight_decay))(channel_input)
        channel_tower = channel_embed
        # Apply Transformer blocks over the channel dimension
        for _ in range(self.transformer_layers):
            attn_output = tf.keras.layers.MultiHeadAttention(
                num_heads=self.num_heads, key_dim=self.embedding_dim, dropout=self.dropout)(
                    channel_tower, channel_tower)
            attn_output = Dropout(self.dropout)(attn_output)
            channel_tower = LayerNormalization(epsilon=1e-6)(channel_tower + attn_output)
            ffn = Dense(self.embedding_dim * 4, activation='relu',
                        kernel_regularizer=l2(self.weight_decay))(channel_tower)
            ffn = Dense(self.embedding_dim, kernel_regularizer=l2(self.weight_decay))(ffn)
            ffn = Dropout(self.dropout)(ffn)
            channel_tower = LayerNormalization(epsilon=1e-6)(channel_tower + ffn)
        # Aggregate over channels
        channel_features = GlobalAveragePooling1D()(channel_tower)
        # Fully connected layer to get the channel-wise feature vector C
        C = Dense(self.embedding_dim, activation='tanh',
                  kernel_regularizer=l2(self.weight_decay))(channel_features)

        # ---- Gating Mechanism ----
        # Concatenate S and C and project to obtain two gating weights.
        gating_input = concatenate([S, C])
        gating = Dense(2, kernel_regularizer=l2(self.weight_decay))(gating_input)
        gating = Activation('softmax')(gating)  # output shape: (batch, 2)
        # Split the gating weights
        g1 = Lambda(lambda x: x[:, 0:1])(gating)  # for step-wise tower
        g2 = Lambda(lambda x: x[:, 1:2])(gating)  # for channel-wise tower
        # Weight each feature vector and then fuse them
        S_weighted = Lambda(lambda x: x[0] * x[1])([S, g1])
        C_weighted = Lambda(lambda x: x[0] * x[1])([C, g2])
        fused = concatenate([S_weighted, C_weighted])

        # ---- Final Classification Layer ----
        outputs = Dense(1, activation='sigmoid', kernel_regularizer=l2(self.weight_decay))(fused)

        model = Model(inputs=inputs, outputs=outputs)
        optimizer = Adam(learning_rate=self.learning_rate)
        model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
        return model

    def summary(self):
        return self.model.summary()

    def fit(self, x, y, epochs=30, batch_size=128, validation_data=None):
        return self.model.fit(x, y, epochs=epochs, batch_size=batch_size, validation_data=validation_data)

    def evaluate(self, x, y):
        return self.model.evaluate(x, y)

    def predict(self, x):
        return self.model.predict(x)

    def save(self, filepath):
        self.model.save(filepath)